# Lecture 08 — Ensemble Models

**Term:** Fall 2025  
**Week/Topic:** Lecture 08  
**Instructor:** Dr. Bushaj  

---

### What you’ll learn
- Bagging, Random Forests, and Boosting (AdaBoost, Gradient Boosting, XGBoost/LightGBM intro).
- Bias–variance tradeoff in ensembles and why ensembles reduce variance.
- Training `RandomForestClassifier` / `RandomForestRegressor`.
- Understanding Boosting: weak learners, sequential fitting, learning rate, number of estimators.
- Avoiding overfitting with tree depth limits, subsampling, feature sampling.
- Evaluating ensembles with cross-validation using accuracy, precision/recall, ROC AUC (classification) and RMSE/MAE/R² (regression).
- Interpreting ensembles (feature importance, permutation importance, partial dependence plots).
- Hyperparameter tuning with `GridSearchCV` / `RandomizedSearchCV`.


## Import Libraries

In [ ]:
#!pip install xgboost

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pandas import ExcelWriter
import plotly.express as px

## Load the data

In [ ]:
my_drive_path = "YOUR_FILE_PATH_HERE"

**I am reading two datasets to show that whichever you run last, all the codes below would still run**

Shows how easy it is to reuse code.

In [ ]:
# Load and prepare the data
bank_df = pd.read_csv(my_drive_path + 'UniversalBank.csv')
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

# Drop any rows with missing values
bank_df = bank_df.dropna()

X = bank_df.drop(columns=['Personal Loan'])
y = bank_df['Personal Loan']


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
#Load flight data:

# Load the dataset from the specified path
delays_df = pd.read_csv(my_drive_path + 'FlightDelays.csv')

# Create an indicator variable for delayed flights
# Using .map for better performance and clarity
delays_df['isDelayed'] = delays_df['Flight Status'].map(lambda status: 1 if status == 'delayed' else 0)

# Convert DAY_WEEK to categorical
delays_df['DAY_WEEK'] = delays_df['DAY_WEEK'].astype('category')

# Create hourly bins for departure time
delays_df['CRS_DEP_TIME'] = (delays_df['CRS_DEP_TIME'] // 100).astype('category')

# Define predictors and outcome
predictors = ['DAY_WEEK', 'CRS_DEP_TIME', 'ORIGIN', 'DEST', 'CARRIER', 'Weather']
outcome = 'isDelayed'

# Create dummy variables for categorical predictors
X = pd.get_dummies(delays_df[predictors], drop_first=True)
y = delays_df[outcome]
classes = ['ontime', 'delayed']

# Split the data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)


## Random Forest Model

### Creating and tuning the model

In [ ]:
# Updated list of hyperparameters for tuning
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 4],
    'max_features': ['sqrt', 'log2']
}

In [ ]:
# GridSearch for the best parameters
rf_model = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy from grid search
best_rf_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)
print("Best Training Accuracy:", grid_search.best_score_)


In [ ]:
# Fit the model with the best parameters on the training data
best_rf_model.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred = best_rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

### Overfitting Plot

In [ ]:
# Overfitting plot: Training vs. Test Score
train_scores = []
test_scores = []

# Extract the best parameters from grid_search, excluding 'n_estimators'
best_params = {k: v for k, v in grid_search.best_params_.items() if k != 'n_estimators'}

# Varying number of estimators to observe overfitting behavior
for n in range(10, 200, 10):
    temp_rf = RandomForestClassifier(n_estimators=n, random_state=42, **best_params)
    temp_rf.fit(X_train, y_train)
    train_scores.append(temp_rf.score(X_train, y_train))
    test_scores.append(temp_rf.score(X_test, y_test))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(10, 200, 10), train_scores, label="Training Accuracy", color="blue")
plt.plot(range(10, 200, 10), test_scores, label="Test Accuracy", color="orange")
plt.xlabel("Number of Estimators")
plt.ylabel("Accuracy")
plt.title("Random Forest Overfitting Plot")
plt.legend()
plt.show()

### Feature Importance

In [ ]:
# Investigating Feature Importance
feature_importances = best_rf_model.feature_importances_
features = X.columns

print(feature_importances)


# Plotting Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importance in Random Forest Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## Extra Trees Model

In [ ]:
# List of hyperparameters for tuning
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [20, 30],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4],
    'max_features': ['sqrt', 'log2']
}

In [ ]:
# GridSearch for the best parameters
et_model = ExtraTreesClassifier(random_state=42)
grid_search = GridSearchCV(estimator=et_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy from grid search
best_et_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)
print("Best Training Accuracy:", grid_search.best_score_)

In [ ]:
# Fit the model with the best parameters on the training data
best_et_model.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred = best_et_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

### Overfitting Plot

In [ ]:
# Overfitting plot: Training vs. Test Score
train_scores = []
test_scores = []

# Extract the best parameters from grid_search, excluding 'n_estimators'
best_params = {k: v for k, v in grid_search.best_params_.items() if k != 'n_estimators'}

# Varying number of estimators to observe overfitting behavior
for n in range(10, 200, 10):
    temp_et = ExtraTreesClassifier(n_estimators=n, random_state=42, **best_params)
    temp_et.fit(X_train, y_train)
    train_scores.append(temp_et.score(X_train, y_train))
    test_scores.append(temp_et.score(X_test, y_test))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(10, 200, 10), train_scores, label="Training Accuracy", color="blue")
plt.plot(range(10, 200, 10), test_scores, label="Test Accuracy", color="orange")
plt.xlabel("Number of Estimators")
plt.ylabel("Accuracy")
plt.title("Extra Trees Overfitting Plot")
plt.legend()
plt.show()




### Feature Importance

In [ ]:
# Investigating Feature Importance
feature_importances = best_et_model.feature_importances_
features = X.columns

# Plotting Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importance in Extra Trees Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## AdaBoost Model

In [ ]:
# List of hyperparameters for tuning
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.5],
    'estimator__max_depth': [1, 3]
}

In [ ]:
# Using Decision Tree as base estimator for AdaBoost
base_estimator = DecisionTreeClassifier(random_state=42)

# GridSearch for the best parameters
ada_model = AdaBoostClassifier(estimator=base_estimator, algorithm='SAMME', random_state=42)
grid_search = GridSearchCV(estimator=ada_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy from grid search
best_ada_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)
print("Best Training Accuracy:", grid_search.best_score_)

# Fit the model with the best parameters on the training data
best_ada_model.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred = best_ada_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

### Overfitting Plot

In [ ]:
# Overfitting plot: Training vs. Test Score
train_scores = []
test_scores = []

# Varying number of estimators to observe overfitting behavior
for n in range(10, 200, 10):
    temp_ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=grid_search.best_params_['estimator__max_depth']),
                                  n_estimators=n,
                                  learning_rate=grid_search.best_params_['learning_rate'],
                                  random_state=42)
    temp_ada.fit(X_train, y_train)
    train_scores.append(temp_ada.score(X_train, y_train))
    test_scores.append(temp_ada.score(X_test, y_test))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(10, 200, 10), train_scores, label="Training Accuracy", color="blue")
plt.plot(range(10, 200, 10), test_scores, label="Test Accuracy", color="orange")
plt.xlabel("Number of Estimators")
plt.ylabel("Accuracy")
plt.title("AdaBoost Overfitting Plot")
plt.legend()
plt.show()


### Feature Importance

In [ ]:
# Investigating Feature Importance
feature_importances = best_ada_model.feature_importances_
features = X.columns

# Plotting Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importance in AdaBoost Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()


## Gradient Boosting Model

In [ ]:
# List of hyperparameters for tuning
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.5],
    'max_depth': [3, 7],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4]
}

In [ ]:
# GridSearch for the best parameters
gb_model = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy from grid search
best_gb_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)
print("Best Training Accuracy:", grid_search.best_score_)

# Fit the model with the best parameters on the training data
best_gb_model.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred = best_gb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

### Overfitting Plot

In [ ]:
# Overfitting plot: Training vs. Test Score
train_scores = []
test_scores = []

# Varying number of estimators to observe overfitting behavior
for n in range(10, 200, 10):
    temp_gb = GradientBoostingClassifier(n_estimators=n,
                                         learning_rate=grid_search.best_params_['learning_rate'],
                                         max_depth=grid_search.best_params_['max_depth'],
                                         min_samples_split=grid_search.best_params_['min_samples_split'],
                                         min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
                                         random_state=42)
    temp_gb.fit(X_train, y_train)
    train_scores.append(temp_gb.score(X_train, y_train))
    test_scores.append(temp_gb.score(X_test, y_test))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(10, 200, 10), train_scores, label="Training Accuracy", color="blue")
plt.plot(range(10, 200, 10), test_scores, label="Test Accuracy", color="orange")
plt.xlabel("Number of Estimators")
plt.ylabel("Accuracy")
plt.title("Gradient Boosting Overfitting Plot")
plt.legend()
plt.show()

### Feature Importance

In [ ]:
# Investigating Feature Importance
feature_importances = best_gb_model.feature_importances_
features = X.columns

# Plotting Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importance in Gradient Boosting Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## XGBoost Model (Optimized Gradient Boosting)

In [ ]:
# List of hyperparameters for tuning
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.5],
    'max_depth': [3, 7],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8,],
    'gamma': [0, 0.1, 0.5]
}


In [ ]:
# GridSearch for the best parameters
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy from grid search
best_xgb_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)
print("Best Training Accuracy:", grid_search.best_score_)

# Fit the model with the best parameters on the training data
best_xgb_model.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred = best_xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

### Overfitting Plot

In [ ]:
# Overfitting plot: Training vs. Test Score
train_scores = []
test_scores = []

# Varying number of estimators to observe overfitting behavior
for n in range(10, 200, 10):
    temp_xgb = XGBClassifier(n_estimators=n,
                             learning_rate=grid_search.best_params_['learning_rate'],
                             max_depth=grid_search.best_params_['max_depth'],
                             subsample=grid_search.best_params_['subsample'],
                             colsample_bytree=grid_search.best_params_['colsample_bytree'],
                             gamma=grid_search.best_params_['gamma'],
                             use_label_encoder=False,
                             eval_metric='logloss',
                             random_state=42)
    temp_xgb.fit(X_train, y_train)
    train_scores.append(temp_xgb.score(X_train, y_train))
    test_scores.append(temp_xgb.score(X_test, y_test))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(10, 200, 10), train_scores, label="Training Accuracy", color="blue")
plt.plot(range(10, 200, 10), test_scores, label="Test Accuracy", color="orange")
plt.xlabel("Number of Estimators")
plt.ylabel("Accuracy")
plt.title("XGBoost Overfitting Plot")
plt.legend()
plt.show()

### Feature Importance

In [ ]:
# Investigating Feature Importance
feature_importances = best_xgb_model.feature_importances_
features = X.columns

# Plotting Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importance in XGBoost Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## Stacking Model Example

In [ ]:
# Define base models for stacking
base_models = [
    ('lr', LogisticRegression(max_iter=500, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
]


In [ ]:
# Define meta-model
meta_model = LogisticRegression(max_iter=2000, random_state=42)

In [ ]:
# Define Stacking Classifier
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

# Fit the stacking model
stacking_model.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred = stacking_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy of Stacking Model:", accuracy)

### Overfitting Plot

In [ ]:
# Overfitting plot: Training vs. Test Score
train_scores = []
test_scores = []

# Varying the number of estimators for the Random Forest and Gradient Boosting components to observe overfitting behavior
for n in range(10, 200, 10):
    temp_rf = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
    temp_gb = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, max_depth=3, random_state=42)

    # Temporarily redefine stacking model with adjusted base models
    temp_stacking_model = StackingClassifier(
        estimators=[('lr', LogisticRegression(max_iter=1000, random_state=42)), ('rf', temp_rf), ('gb', temp_gb)],
        final_estimator=meta_model,
        cv=5
    )
    temp_stacking_model.fit(X_train, y_train)
    train_scores.append(temp_stacking_model.score(X_train, y_train))
    test_scores.append(temp_stacking_model.score(X_test, y_test))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(10, 200, 10), train_scores, label="Training Accuracy", color="blue")
plt.plot(range(10, 200, 10), test_scores, label="Test Accuracy", color="orange")
plt.xlabel("Number of Estimators for RF/GB in Stacking")
plt.ylabel("Accuracy")
plt.title("Stacking Model Overfitting Plot")
plt.legend()
plt.show()


### Feature Importance
** Note: Feature importance in stacking can be challenging to interpret, as each base model may have different feature importances.**
** Here, we can investigate feature importances in Random Forest or Gradient Boosting if desired, but overall stacking feature importance is not directly available.**
** Example: Investigate feature importance from the Random Forest component**

In [ ]:
best_rf_model = stacking_model.named_estimators_['rf']
feature_importances = best_rf_model.feature_importances_
features = X.columns

# Plotting Feature Importance for Random Forest component
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importance in Random Forest Component of Stacking Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## Combined Analysis

In [ ]:
# Function to calculate and display evaluation metrics
def evaluate_metrics(y_test, y_pred, y_pred_proba=None):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None

    metrics = {
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1 Score': [f1],
        'AUC': [auc] if auc is not None else ['N/A']
    }
    # Print metrics to console
    print(pd.DataFrame(metrics))
    return pd.DataFrame(metrics)

# Function to display and return confusion matrix as a DataFrame
def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.show()
    return pd.DataFrame(cm, index=['Actual No', 'Actual Yes'], columns=['Predicted No', 'Predicted Yes'])

# Function to plot ROC curve
def plot_roc_curve(y_test, y_pred_proba):
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color="blue", label=f"AUC = {auc:.2f}")
    plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()

# Function to display and return feature importance as a DataFrame (for models with feature_importances_)
def plot_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        feature_importances = model.feature_importances_
        sorted_idx = np.argsort(feature_importances)[::-1]  # Sort features by importance

        plt.figure(figsize=(10, 6))
        sns.barplot(x=feature_importances[sorted_idx], y=feature_names[sorted_idx])
        plt.title("Feature Importance")
        plt.xlabel("Importance Score")
        plt.ylabel("Features")
        plt.show()

        importance_df = pd.DataFrame({
            'Feature': feature_names[sorted_idx],
            'Importance': feature_importances[sorted_idx]
        })
        return importance_df
    else:
        print("Feature importance is not available for this model.")
        return pd.DataFrame({'Feature': [], 'Importance': []})  # Empty DataFrame if not available

In [ ]:
# Dictionary of trained models for analysis
models = {
    "Random Forest": best_rf_model,
    "Extra Trees": best_et_model,
    "AdaBoost": best_ada_model,
    "Gradient Boosting": best_gb_model,
    "XGBoost": best_xgb_model,
    "Stacking Model": stacking_model
}

# Initialize an Excel writer
with ExcelWriter("Model_Analysis.xlsx", engine="openpyxl") as writer:
    # Perform analysis for each model and save to different sheets
    for model_name, model in models.items():
        print(f"\nAnalyzing {model_name} Results")

        # Get predictions and predicted probabilities if available
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

        # 1. Evaluation Metrics
        print("Evaluation Metrics:")
        metrics_df = evaluate_metrics(y_test, y_pred, y_pred_proba)
        metrics_df.to_excel(writer, sheet_name=f"{model_name} Metrics", index=False)

        # 2. Confusion Matrix
        print("\nConfusion Matrix:")
        cm_df = plot_confusion_matrix(y_test, y_pred)
        cm_df.to_excel(writer, sheet_name=f"{model_name} Confusion Matrix", index=True)

        # 3. ROC Curve (if probabilities are available)
        if y_pred_proba is not None:
            print("\nROC Curve:")
            plot_roc_curve(y_test, y_pred_proba)

        # 4. Feature Importance (if available)
        print("\nFeature Importance:")
        feature_importance_df = plot_feature_importance(model, X.columns)
        if not feature_importance_df.empty:
            feature_importance_df.to_excel(writer, sheet_name=f"{model_name} Feature Importance", index=False)

In [ ]:
# Dictionary of trained models for analysis
models = {
    "Random Forest": best_rf_model,
    "Extra Trees": best_et_model,
    "AdaBoost": best_ada_model,
    "Gradient Boosting": best_gb_model,
    "XGBoost": best_xgb_model,
    "Stacking Model": stacking_model
}

# Perform analysis for each model
for model_name, model in models.items():
    print(f"\nAnalyzing {model_name} Results")

    # Get predictions and predicted probabilities if available
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # Evaluate metrics
    print("Evaluation Metrics:")
    evaluate_metrics(y_test, y_pred, y_pred_proba)

    # Plot confusion matrix
    print("\nConfusion Matrix:")
    plot_confusion_matrix(y_test, y_pred)

    # Plot ROC curve if probabilities are available
    if y_pred_proba is not None:
        print("\nROC Curve:")
        plot_roc_curve(y_test, y_pred_proba)

    # Plot feature importance if available
    print("\nFeature Importance:")
    plot_feature_importance(model, X.columns)


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Dictionary of trained models for analysis
models = {
    "Random Forest": best_rf_model,
    "Extra Trees": best_et_model,
    "AdaBoost": best_ada_model,
    "Gradient Boosting": best_gb_model,
    "XGBoost": best_xgb_model,
    "Stacking Model": stacking_model
}

# Initialize a plot
plt.figure(figsize=(10, 8))

# Colors for each model in the plot
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']

# Loop through each model, calculate ROC curve and AUC, then plot
for (model_name, model), color in zip(models.items(), colors):
    # Check if the model has a predict_proba method
    if hasattr(model, "predict_proba"):
        y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability for the positive class
    else:
        # For models that do not support predict_proba, use decision_function
        y_pred_proba = model.decision_function(X_test)

    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)

    # Plot ROC curve
    plt.plot(fpr, tpr, color=color, label=f"{model_name} (AUC = {auc:.2f})")

# Plot the diagonal line for random guessing
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")

# Set plot title and labels
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Models")
plt.legend(loc="lower right")
plt.show()
